In [ ]:
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()

# Find 'src' by walking up the directory tree
src_path = None
for parent in [current_dir] + list(current_dir.parents):
    candidate = parent / 'src'
    if candidate.exists() and candidate.is_dir():
        src_path = candidate
        break

if src_path:
    if str(src_path) not in sys.path:
        sys.path.append(str(src_path))
    print(f"Success: Linked to src at {src_path}")
    
    # Import
    try:
        from config import *
        print(f"Environment linked. Data dir: {DATA_DIR}")    
    except ImportError as e:
        print(f"CRITICAL: Found src but could not import config. {e}")
else:
    print("CRITICAL ERROR: Could not find 'src' directory in any parent folder.")
    print(f"Current search path: {current_dir}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from plotting import *
setup_plotting_style()

In [ ]:
# Imports and env setup
from analysis import *
import pandas as pd
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

NB_ID = "21"

In [ ]:
# load dataset
print(f"Loading dataset from: {MACHINE_A_DATASET_FILE}")
df = pd.read_csv(MACHINE_A_DATASET_FILE)

features_to_plot = MACHINE_A_FEATURES
print(f"Visualizing Features: {features_to_plot}")

print(f"Dataset Shape: {df.shape}")
df.head(3)

In [ ]:
# Visualizes class balance and feature spreads.
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='label', data=df, hue='label', palette=['C0', 'C1'], legend=False)

# Add counts on top
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        center_x = p.get_x() + p.get_width() / 2
        ax.annotate(f'{int(height)}', 
                    (center_x, height),     # X = center, Y = top of bar
                    ha='center',            # Center the text horizontally
                    va='bottom',            # Sit the text on top of the coordinate
                    xytext=(0, 0),          # Lift it 5 points up so it doesn't touch the bar
                    textcoords='offset points')

current_ymax = plt.ylim()[1]
plt.ylim(0, current_ymax * 1.15)
plt.title("Class Distribution (0=Clean, 1=Jamming)")
save_plot("class_distribution",machine_id="A",nb_id=NB_ID,fig_id="01")
plt.show()

# Feature Distributions (KDE)
num_features = len(features_to_plot)
cols = 2
rows = (num_features + 1) // cols

plt.figure(figsize=(15, 5 * rows))
for i, feature in enumerate(features_to_plot):
    plt.subplot(rows, cols, i + 1)
    sns.kdeplot(data=df, x=feature, hue='label', fill=True, 
                palette={0: 'C0', 1: 'C1'}, common_norm=False)
    plt.title(f"Distribution of {feature}")

plt.tight_layout()
save_plot("feature_distributions",machine_id="A",nb_id=NB_ID,fig_id="02")
plt.show()

## Strategy Note: Handling Class Imbalance

**Observation:**
The dataset contains a significant imbalance:
* **Jammed (Label 1):** ~300,000 samples (67%)
* **Clean (Label 0):** ~150,000 samples (33%)

**Implication:**
A naive model could achieve 67% accuracy simply by predicting "Jammed" for every signal. This would result in poor detection of actual clean signals.

**Current Approach:**
* We are currently relying on **Cost-Sensitive Learning** (`class_weight='balanced'`) to handle the 2:1 imbalance.
* Preliminary tests show this is sufficient to achieve >0.93 F1 score.

**Deferred Strategies (Future Work):**
If the model exhibits poor recall on the minority class ("Clean" signals) during full training, we will revisit this stage and implement:
1.  **SMOTE (Synthetic Minority Over-sampling Technique):** To generate synthetic "Clean" samples in the feature space.
2.  **Threshold Moving:** Adjusting the decision threshold (currently 0.5) to favor the minority class.
3.  **Signal Augmentation:** (Advanced) Going back to the raw `.npy` data to inject phase noise or fading into the clean signals.

**Decision:**
* **Pause:** Do not implement SMOTE or signal augmentation yet.

In [ ]:
# Feature Interaction Matrix (Pair Plot)

# We sample 2000 points to keep rendering fast
plot_sample = df.sample(n=min(2000, len(df)), random_state=42)

# Create the plot
g = sns.pairplot(
    plot_sample, 
    vars=features_to_plot, 
    hue='label', 
    palette={0: 'C0', 1: 'C1'},
    plot_kws={'alpha': 0.6, 's': 15}, # s=size of dots, alpha=transparency
    diag_kind='kde' # Curves on the diagonal
)

g.fig.suptitle("Feature Interaction Matrix", y=1.02)

# Save it
save_plot("interaction_matrix",machine_id="A",nb_id=NB_ID,fig_id="03")
plt.show()

In [ ]:
# Checks metadata consistency and feature relationships.

# Sensitivity Check (Metadata vs Calculated Power)
if 'jamming_power' in df.columns:
    # Filter for valid jamming samples
    jamming_df = df[(df['label'] == 1) & (df['jamming_power'] >= 0)]
    
    if len(jamming_df) > 0:
        plt.figure(figsize=(12, 5))
        y_feature = 'Mean Power' if 'Mean Power' in features_to_plot else features_to_plot[0]
        
        sns.boxplot(x='jamming_power', y=y_feature, data=jamming_df, color='C1')
        plt.title(f"Sensitivity Check: {y_feature} vs. Jamming Power")
        
        save_plot("sensitivity_check",machine_id="A",nb_id=NB_ID,fig_id="04")
        plt.show()

# Correlation Matrix
plt.figure(figsize=(10, 8))
corr = df[features_to_plot].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", vmin=-1, vmax=1)
plt.title("Feature Correlation Matrix")
save_plot("correlation_matrix",machine_id="A",nb_id=NB_ID,fig_id="05")
plt.show()

In [ ]:
# Experiment 1 - Feature Selection (VIF & Ablation)

# Check Multicollinearity (VIF)
print("--- Variance Inflation Factors (VIF) ---")
vif_df = calc_vif(df[features_to_plot])
print(vif_df)
print("\n")

print("--- Running Ablation Study ---")
ablation_results = run_ablation_study(
    df, 
    df['label'], 
    features_to_plot, 
    features_to_plot
)

plt.figure(figsize=(10, 6))

# Bar Chart
sns.barplot(
    data=ablation_results, 
    x='F1 Score', 
    y='Scenario', 
    hue='Scenario',
    palette='viridis', 
    legend=False
)

# Error Bars (Visualizing Stability)
plt.errorbar(
    x=ablation_results['F1 Score'], 
    y=np.arange(len(ablation_results)), 
    xerr=ablation_results['Std Dev'], 
    fmt='none', 
    c='black', 
    capsize=5
)

# Aesthetics
min_score = ablation_results['F1 Score'].min() - 0.02
max_score = ablation_results['F1 Score'].max() + 0.01
plt.xlim(min_score, max_score)

plt.title("Impact of Feature Removal on Model Performance")
plt.xlabel("F1 Score (Mean +/- Std Dev)")
plt.grid(axis='x', linestyle='--', alpha=0.7)

save_plot("feature_ablation_results",machine_id="A",nb_id=NB_ID,fig_id="06")
plt.show()

## Experiment: Feature Selection & Multicollinearity

**Objective:**
To identify and remove redundant features that cause model instability. We defined "redundant" as features with a Variance Inflation Factor (VIF) > 10 or high pairwise correlation.

**1. Data Analysis:**
* **VIF Check:** We detected severe multicollinearity in two features:
    * 🚨 **Spectral Entropy:** VIF `554.06` (Extremely High)
    * 🚨 **Variance:** VIF `405.70` (Extremely High)
* **Correlation Matrix:**
    * *Spectral Entropy* vs. *Spectral Flatness* showed strong correlation ($r=0.78$).
    * *Variance* showed moderate-to-high correlation with *Mean Power* ($r=0.65$), contributing to the shared variance.

**2. Ablation Study (Performance Check):**
We tested removing these "suspect" features to see if the model lost predictive power.
* **Baseline (All 7 Features):** F1 Score `0.9392`
* **Remove Spectral Entropy:** F1 Score `0.9395` (⬆️ **Improved**)
    * *Insight:* Removing Entropy actually increased performance, confirming it was just adding noise.
* **Remove Variance:** F1 Score `0.9289` (⬇️ Slight Drop)
    * *Insight:* While Variance has some unique signal, its VIF of 406 makes it dangerous for stability. Since *Mean Power* covers similar physical properties with a lower VIF (159), we prioritize Mean Power.

**Decision:**
To ensure a robust model, we are **removing**:
1.  ❌ `Spectral Entropy` (Redundant with Flatness, improved score when removed).
2.  ❌ `Variance` (Extreme VIF, redundant with Mean Power).

**Final Feature Set:**
`Mean Power`, `PAPR`, `Kurtosis`, `Skewness`, `Spectral Flatness`.

In [ ]:
# Experiment 2 - Scaler Selection

# Visual Comparison
check_feats = ['Mean Power', 'Kurtosis'] 
scalers = {
    'StandardScaler': StandardScaler(), 
    'MinMaxScaler': MinMaxScaler()
}

scaler_colors = {
    'StandardScaler': 'C0',
    'MinMaxScaler':   'C1'
}

plt.figure(figsize=(14, 6))
plot_idx = 1

for feature in check_feats:
    raw_data = df[[feature]].values

    for name, scaler in scalers.items():
        plt.subplot(2, 2, plot_idx)

        # Transform
        scaled_data = scaler.fit_transform(raw_data)

        sns.histplot(scaled_data, bins=50, kde=True, legend=False, 
                     color=scaler_colors[name])
        
        plt.title(f"{feature} using {name}")
        plot_idx += 1

plt.tight_layout()
save_plot("scaler_comparison",machine_id="A",nb_id=NB_ID,fig_id="07")
plt.show()

# 2. Performance Comparison
print(f"\n{'SCALER':<20} | {'F1 SCORE':<10} | {'STD DEV'}")
print("-" * 50)

for name, scaler in scalers.items():
    pipe = make_pipeline(scaler, LogisticRegression(class_weight='balanced', max_iter=1000))
    scores = cross_val_score(pipe, df[features_to_plot], df['label'], cv=5, scoring='f1')
    print(f"{name:<20} | {np.mean(scores):.4f}     | +/-{np.std(scores):.4f}")

## Decision: Scaler Selection

**Objective:**
To determine the optimal scaling technique for the RF feature set, comparing `StandardScaler` (Z-score normalization) against `MinMaxScaler` (0-1 scaling).

**Results:**
A 5-fold cross-validation comparison yielded the following F1 scores:
* **StandardScaler:** `0.9284` (+/- 0.0005)
* **MinMaxScaler:** `0.9247` (+/- 0.0006)

**Conclusion:**
The performance difference between the two scalers is statistically negligible (< 0.001).
* **Decision:** We will proceed with **`StandardScaler`**.
* **Rationale:** `StandardScaler` is the industry standard for Logistic Regression (which assumes normally distributed features). It naturally handles the negative values in our data (e.g., *Skewness*, *Kurtosis*) and is less sensitive to the specific outlier bounds found in the *Mean Power* feature compared to MinMax.